# Multi-Agent Self-Play Training

All horses share one policy and compete simultaneously. Continues from
phased single-agent training (SB3) by transferring weights into RLlib.

**Reward phases** (same as single-agent):
- **Phase 1** — Race & Kick
- **Phase 2** — +Cornering
- **Phase 3** — +Archetype/Skills

**Curriculum stages** (per phase):
1. **Oval** (4 horses) — self-play basics
2. **All tracks** (4-8 horses) — generalize
3. **Large fields** (6-12 horses) — positioning & overtaking

**How to use:**
1. Set `REWARD_PHASE`, `STAGE`, and optionally `SB3_MODEL_PATH`
2. Run all cells top to bottom
3. If runtime disconnects, reconnect and run all — auto-resumes from Drive

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — edit these before running
# ============================================================

# Reward phase: 1=race/kick only, 2=+cornering, 3=+archetype/skills
REWARD_PHASE = 2

# Training stage: 1=oval, 2=all tracks, 3=large fields
STAGE = 1

# Stage configs
STAGE_CONFIGS = {
    1: {
        "tracks": ["tracks/test_oval.json"],
        "min_horses": 4,
        "max_horses": 4,
        "timesteps": 2_000_000,
        "description": "Oval — self-play basics",
    },
    2: {
        "tracks": None,  # None = all tracks
        "min_horses": 4,
        "max_horses": 8,
        "timesteps": 3_000_000,
        "description": "All tracks — generalize",
    },
    3: {
        "tracks": None,
        "min_horses": 6,
        "max_horses": 12,
        "timesteps": 5_000_000,
        "description": "Large fields — positioning & overtaking",
    },
}

# SB3 pretrained model path — set to .zip path for initial weight transfer
# Leave None to start fresh or resume from RLlib checkpoint
SB3_MODEL_PATH = None  # e.g., "/content/drive/MyDrive/hr-checkpoints/phase_2/stage_3/final_model.zip"

# BT opponent ratio: fraction of horses controlled by BT (0 = pure self-play)
BT_OPPONENT_RATIO = 0.0

# Checkpoint dir on Google Drive (organized by phase)
DRIVE_BASE = "/content/drive/MyDrive/hr-checkpoints/multi_agent"
DRIVE_DIR = f"{DRIVE_BASE}/phase_{REWARD_PHASE}"

# Resume: 'auto' finds latest RLlib checkpoint, None starts fresh
RESUME_CHECKPOINT = 'auto'

# Learning rate — lower for later phases
LR_BY_PHASE = {1: 3e-4, 2: 1e-4, 3: 5e-5}

# Entropy annealing
ENTROPY_START = 0.02
ENTROPY_END = 0.005

# Logging
LOG_EVERY = 5
SAVE_EVERY = 25

## Setup

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update repo
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Install dependencies (both RLlib and SB3 — SB3 needed for weight transfer)
!pip install -q "ray[rllib]>=2.9.0" "gymnasium>=1.0.0" "pettingzoo>=1.24.0" "torch>=2.0.0" "onnx>=1.21.0" "onnxruntime>=1.24.4" "onnxscript>=0.6.2" "stable-baselines3>=2.3.0" psutil

# Remove old gym package
!pip uninstall -q -y gym 2>/dev/null || true

# Install the project in editable mode
!pip install -q -e .

import warnings
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

## Resolve Stage Config & Checkpoint

In [ ]:
import glob
from pathlib import Path

cfg = STAGE_CONFIGS[STAGE]
stage_dir = os.path.join(DRIVE_DIR, f"stage_{STAGE}")
os.makedirs(stage_dir, exist_ok=True)

lr = LR_BY_PHASE[REWARD_PHASE]

# Resolve tracks
if cfg["tracks"] is None:
    track_paths = sorted(glob.glob("tracks/*.json"))
else:
    track_paths = cfg["tracks"]

print(f"Reward Phase {REWARD_PHASE}: {'Race/Kick' if REWARD_PHASE == 1 else '+Cornering' if REWARD_PHASE == 2 else '+Archetype/Skills'}")
print(f"Stage {STAGE}: {cfg['description']}")
print(f"Tracks: {len(track_paths)} ({', '.join(Path(t).stem for t in track_paths)})")
print(f"Horses: {cfg['min_horses']}-{cfg['max_horses']}")
print(f"Timesteps: {cfg['timesteps']:,}")
print(f"Learning rate: {lr}")
print(f"BT opponent ratio: {BT_OPPONENT_RATIO}")
print(f"Checkpoint dir: {stage_dir}")
print()

# Find checkpoint to resume from
checkpoint_path = None
use_sb3_transfer = False

def find_latest_rllib_checkpoint(checkpoint_dir: str) -> str | None:
    """Find the most recent RLlib checkpoint."""
    markers = []
    for pattern in ["**/algorithm_state.pkl", "**/.is_checkpoint"]:
        markers.extend(glob.glob(os.path.join(checkpoint_dir, pattern), recursive=True))
    if not markers:
        return None
    markers.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return str(Path(markers[0]).parent)

if RESUME_CHECKPOINT == 'auto':
    # Look for existing RLlib checkpoint in current stage
    checkpoint_path = find_latest_rllib_checkpoint(stage_dir)
    if checkpoint_path:
        print(f"Auto-resume: {checkpoint_path}")
    elif STAGE > 1:
        prev_cp = find_latest_rllib_checkpoint(os.path.join(DRIVE_DIR, f"stage_{STAGE-1}"))
        if prev_cp:
            checkpoint_path = prev_cp
            print(f"Loading previous stage: {checkpoint_path}")
    elif REWARD_PHASE > 1:
        prev_phase_dir = f"{DRIVE_BASE}/phase_{REWARD_PHASE - 1}"
        for s in range(max(STAGE_CONFIGS.keys()), 0, -1):
            prev_cp = find_latest_rllib_checkpoint(os.path.join(prev_phase_dir, f"stage_{s}"))
            if prev_cp:
                checkpoint_path = prev_cp
                print(f"Loading from Phase {REWARD_PHASE - 1} Stage {s}: {checkpoint_path}")
                break

    # If no RLlib checkpoint and SB3 model provided, use weight transfer
    if not checkpoint_path and SB3_MODEL_PATH:
        use_sb3_transfer = True
        print(f"No RLlib checkpoint found — will transfer from SB3: {SB3_MODEL_PATH}")
    elif not checkpoint_path:
        print("No checkpoint found — starting fresh")
elif RESUME_CHECKPOINT:
    checkpoint_path = RESUME_CHECKPOINT
    print(f"Resuming from: {checkpoint_path}")
else:
    if SB3_MODEL_PATH:
        use_sb3_transfer = True
        print(f"Starting from SB3 weights: {SB3_MODEL_PATH}")
    else:
        print("Starting fresh (RESUME_CHECKPOINT=None)")

## SB3→RLlib Weight Transfer + Training

In [ ]:
import math
import time
import json
import torch
import psutil
import numpy as np
import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.policy.policy import PolicySpec

from horse_racing.rllib_env import HorseRacingRLlibEnv

# ---------------------------------------------------------------------------
# SB3 → RLlib weight transfer
# ---------------------------------------------------------------------------

TANG_SCALE = 4.5
NORM_SCALE = 3.0


def transfer_sb3_to_rllib(sb3_path: str, algo, policy_id: str = "shared_policy"):
    """Load SB3 PPO weights into RLlib model with action scaling.

    SB3 outputs [-1, 1] then remaps ×[4.5, 3.0]. RLlib outputs raw physics
    values. The action output layer is scaled to produce identical actions.
    """
    from stable_baselines3 import PPO as SB3_PPO

    sb3_model = SB3_PPO.load(sb3_path)
    sb3_sd = sb3_model.policy.state_dict()
    policy = algo.get_policy(policy_id)
    rllib_sd = policy.model.state_dict()

    new_sd = {}

    # Actor hidden layers
    new_sd["_hidden_layers.0._model.0.weight"] = sb3_sd["mlp_extractor.policy_net.0.weight"].clone()
    new_sd["_hidden_layers.0._model.0.bias"] = sb3_sd["mlp_extractor.policy_net.0.bias"].clone()
    new_sd["_hidden_layers.1._model.0.weight"] = sb3_sd["mlp_extractor.policy_net.2.weight"].clone()
    new_sd["_hidden_layers.1._model.0.bias"] = sb3_sd["mlp_extractor.policy_net.2.bias"].clone()

    # Value hidden layers
    new_sd["_value_branch_separate.0._model.0.weight"] = sb3_sd["mlp_extractor.value_net.0.weight"].clone()
    new_sd["_value_branch_separate.0._model.0.bias"] = sb3_sd["mlp_extractor.value_net.0.bias"].clone()
    new_sd["_value_branch_separate.1._model.0.weight"] = sb3_sd["mlp_extractor.value_net.2.weight"].clone()
    new_sd["_value_branch_separate.1._model.0.bias"] = sb3_sd["mlp_extractor.value_net.2.bias"].clone()

    # Value head
    new_sd["_value_branch._model.0.weight"] = sb3_sd["value_net.weight"].clone()
    new_sd["_value_branch._model.0.bias"] = sb3_sd["value_net.bias"].clone()

    # Action output: RLlib _logits is (4, 256) — rows [0:2] means, [2:4] log_std
    logits_weight = torch.zeros(4, 256)
    logits_bias = torch.zeros(4)

    logits_weight[0] = sb3_sd["action_net.weight"][0] * TANG_SCALE
    logits_weight[1] = sb3_sd["action_net.weight"][1] * NORM_SCALE
    logits_bias[0] = sb3_sd["action_net.bias"][0] * TANG_SCALE
    logits_bias[1] = sb3_sd["action_net.bias"][1] * NORM_SCALE

    sb3_log_std = sb3_sd["log_std"]
    logits_weight[2:] = 0.0
    logits_bias[2] = sb3_log_std[0] + math.log(TANG_SCALE)
    logits_bias[3] = sb3_log_std[1] + math.log(NORM_SCALE)

    new_sd["_logits._model.0.weight"] = logits_weight
    new_sd["_logits._model.0.bias"] = logits_bias

    policy.model.load_state_dict(new_sd)
    policy.set_weights(policy.get_weights())

    # Quick verification
    obs = torch.randn(10, 111, dtype=torch.float32)
    with torch.no_grad():
        feat = sb3_model.policy.mlp_extractor.forward_actor(obs)
        raw = sb3_model.policy.action_net(feat)
        sb3_actions = torch.stack([raw[:, 0] * TANG_SCALE, raw[:, 1] * NORM_SCALE], dim=1)
        model_out, _ = policy.model({"obs": obs})
        rllib_actions = model_out[:, :2]
    max_diff = (sb3_actions - rllib_actions).abs().max().item()
    print(f"  Weight transfer verification: max action diff = {max_diff:.2e} {'(OK)' if max_diff < 1e-5 else '(WARNING)'}")
    return max_diff < 1e-5


# ---------------------------------------------------------------------------
# Hardware detection
# ---------------------------------------------------------------------------

num_gpus = 1 if torch.cuda.is_available() else 0
total_cpus = psutil.cpu_count()
num_workers = max(1, min(total_cpus - 1, 6))

print(f"Hardware: {num_workers} CPU workers, {num_gpus} GPU")
print(f"Tracks: {len(track_paths)}")
print(f"Target: {cfg['timesteps']:,} timesteps")
print()

# ---------------------------------------------------------------------------
# Init Ray + build algorithm
# ---------------------------------------------------------------------------

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

env_config = {
    "track_paths": track_paths,
    "min_horse_count": cfg["min_horses"],
    "max_horse_count": cfg["max_horses"],
    "max_steps": 5000,
    "randomize_archetypes": True,
    "randomize_genomes": True,
    "bt_opponent_ratio": BT_OPPONENT_RATIO,
    "reward_phase": REWARD_PHASE,
}

config = (
    PPOConfig()
    .api_stack(
        enable_rl_module_and_learner=False,
        enable_env_runner_and_connector_v2=False,
    )
    .environment(
        env=HorseRacingRLlibEnv,
        env_config=env_config,
    )
    .env_runners(
        num_env_runners=num_workers,
        num_envs_per_env_runner=1,
    )
    .training(
        train_batch_size=16000,
        sgd_minibatch_size=512,
        num_sgd_iter=10,
        lr=lr,
        gamma=0.998,
        lambda_=0.95,
        clip_param=0.2,
        vf_clip_param=50.0,
        entropy_coeff=ENTROPY_START,
        model={
            "fcnet_hiddens": [256, 256],
            "fcnet_activation": "tanh",
        },
    )
    .resources(
        num_gpus=num_gpus,
    )
    .framework("torch")
    .multi_agent(
        policies={"shared_policy": PolicySpec()},
        policy_mapping_fn=lambda agent_id, *args, **kwargs: "shared_policy",
    )
)

algo = config.build()

# Restore from RLlib checkpoint or transfer SB3 weights
if checkpoint_path:
    algo.restore(checkpoint_path)
    print(f"Restored from: {checkpoint_path}")
elif use_sb3_transfer:
    print(f"Transferring SB3 weights from: {SB3_MODEL_PATH}")
    transfer_sb3_to_rllib(SB3_MODEL_PATH, algo)
    # Save initial checkpoint so we can resume later
    init_save = algo.save(os.path.join(stage_dir, "sb3_transfer"))
    print(f"Initial checkpoint saved: {init_save}")
else:
    print("Starting with random weights")

print("\nAlgorithm ready.")

In [ ]:
# --- Training loop with entropy annealing ---
os.makedirs(stage_dir, exist_ok=True)
target_timesteps = cfg["timesteps"]

best_reward = float("-inf")
total_timesteps = 0
iterations = 0
start_time = time.time()
history = []

print(f"Training Phase {REWARD_PHASE} / Stage {STAGE}: {cfg['description']}")
print(f"Target: {target_timesteps:,} timesteps | LR: {lr}")
print(f"Entropy: {ENTROPY_START} → {ENTROPY_END}")
print(f"Checkpoints → {stage_dir}\n")

try:
    while total_timesteps < target_timesteps:
        # Entropy annealing
        progress = min(1.0, total_timesteps / target_timesteps)
        new_ent = ENTROPY_END + (ENTROPY_START - ENTROPY_END) * (1.0 - progress)
        algo.config["entropy_coeff"] = new_ent

        result = algo.train()
        iterations += 1

        # Extract metrics
        er = result.get("env_runners", result.get("sampler_results", {}))
        mean_reward = er.get("episode_return_mean", er.get("episode_reward_mean", 0.0))
        max_reward = er.get("episode_return_max", er.get("episode_reward_max", 0.0))
        ep_len = er.get("episode_len_mean", 0.0)
        num_episodes = er.get("num_episodes", 0)
        total_timesteps = result.get(
            "num_env_steps_sampled_lifetime",
            result.get("timesteps_total", 0),
        )

        history.append({
            "iter": iterations,
            "timesteps": total_timesteps,
            "mean_reward": float(mean_reward),
            "max_reward": float(max_reward),
            "ep_len": float(ep_len),
            "episodes": int(num_episodes),
            "entropy_coeff": float(new_ent),
        })

        if iterations % LOG_EVERY == 0 or iterations == 1:
            elapsed = time.time() - start_time
            rate = total_timesteps / elapsed if elapsed > 0 else 0
            pct = total_timesteps / target_timesteps * 100
            print(
                f"  [{pct:5.1f}%] Iter {iterations:4d} | "
                f"ts={total_timesteps:>10,} | "
                f"reward={mean_reward:8.1f} (max={max_reward:8.1f}) | "
                f"ep_len={ep_len:5.0f} | "
                f"ent={new_ent:.4f} | "
                f"{rate:,.0f} ts/s"
            )

        if mean_reward > best_reward:
            best_reward = mean_reward
            algo.save(os.path.join(stage_dir, "best"))
            if iterations % LOG_EVERY == 0:
                print(f"  *** New best: {best_reward:.1f}")

        if iterations % SAVE_EVERY == 0:
            algo.save(os.path.join(stage_dir, f"iter_{iterations}"))
            print(f"  Checkpoint saved: iter_{iterations}")
            with open(os.path.join(stage_dir, "history.json"), "w") as f:
                json.dump(history, f)

except KeyboardInterrupt:
    print("\nTraining interrupted.")

# Final save
algo.save(os.path.join(stage_dir, "latest"))
with open(os.path.join(stage_dir, "history.json"), "w") as f:
    json.dump(history, f)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"Phase {REWARD_PHASE} / Stage {STAGE} complete")
print(f"  Timesteps: {total_timesteps:,}")
print(f"  Iterations: {iterations}")
print(f"  Best reward: {best_reward:.1f}")
print(f"  Time: {elapsed/60:.1f} min")
print(f"  Checkpoint: {stage_dir}/latest")
print(f"{'='*60}")

## Export to ONNX

In [ ]:
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
from horse_racing.types import OBS_SIZE


class RLlibPolicyWrapper(nn.Module):
    """Wraps RLlib model for ONNX export: obs -> action means."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, obs):
        model_out, _ = self.model({"obs": obs})
        return model_out[:, :2]  # action means only


policy = algo.get_policy("shared_policy")
model = policy.model
model.eval()

wrapper = RLlibPolicyWrapper(model).cpu()
wrapper.eval()

onnx_path = os.path.join(stage_dir, f"stage_{STAGE}_jockey.onnx")
dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)

with torch.no_grad():
    test_out = wrapper(dummy)
    print(f"Test output: {test_out.numpy()}")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"],
    output_names=["action"],
    dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
result = session.run(None, {"obs": np.zeros((1, OBS_SIZE), dtype=np.float32)})
print(f"ONNX output: {result[0]}")
print(f"Exported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")

## Quick Eval Race

In [ ]:
# Run a quick self-play race using the trained model
from horse_racing.engine import EngineConfig, HorseRacingEngine
from horse_racing.types import HorseAction

eval_horses = cfg["max_horses"]
session = ort.InferenceSession(onnx_path)
engine = HorseRacingEngine("tracks/tokyo.json", EngineConfig(horse_count=eval_horses))

for step in range(5000):
    obs_list = engine.get_observations()
    obs_arrays = np.array([engine.obs_to_array(o) for o in obs_list], dtype=np.float32)
    actions = session.run(None, {"obs": obs_arrays})[0]

    action_list = [HorseAction(float(a[0]), float(a[1])) for a in actions]
    engine.step(action_list)

    if all(h.finished for h in engine.horses):
        break

placements = engine.get_placements()
print(f"Self-play race finished in {step+1} steps\n")
print(f"{'Horse':>8} {'Place':>6} {'Progress':>10} {'Stamina':>10} {'Finished':>10}")
print("-" * 50)
for i, hs in enumerate(engine.horses):
    stam = hs.runtime.current_stamina / hs.base_attrs.stamina if hs.base_attrs.stamina > 0 else 0
    print(f"  H{i:>5} {placements[i]:>6} {hs.track_progress:>10.3f} {stam:>10.2f} {'Yes' if hs.finished else 'No':>10}")

algo.stop()
ray.shutdown()
print("\nDone.")